# C — Angle 2 Random Forest

**Before running:** use the C environment (`scikit-learn 1.9.1`), see `START_HERE.md`.

The full retrain is about 10 minutes and writes into `C_angle2_forest/outputs/`. The included outputs are precomputed; run the full chain only if you intend to replace them.

In [ ]:
from pathlib import Path
import sys, subprocess, os

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "README.md").exists():
    ROOT = ROOT.parent
assert (ROOT / "README.md").exists(), "Open this notebook from inside the repository."
print("Repository root:", ROOT)
print("Python:", sys.version.split()[0])
assert sys.version_info >= (3, 12), "Use Python 3.12 (64-bit)."

def run(args, cwd=None):
    cmd = [sys.executable, *map(str, args)]
    where = ROOT if cwd is None else ROOT / cwd
    print("RUN:", " ".join(cmd))
    print("CWD:", where)
    subprocess.run(cmd, cwd=where, check=True)


In [ ]:
for name in ["data/cleaned.csv", "data/split_manifest.csv"]:
    p = ROOT / name
    assert p.exists(), f"Missing {name}"
    print("OK", name, p.stat().st_size, "bytes")


In [ ]:
import numpy, pandas, sklearn
print("numpy", numpy.__version__)
print("pandas", pandas.__version__)
print("scikit-learn", sklearn.__version__)

print("Expected C scikit-learn: 1.9.1")
if sklearn.__version__ != "1.9.1":
    print("WARNING: recorded results used 1.9.1; re-run numbers may differ.")

## Inspect the included results (no training)

In [ ]:
import pandas as pd
print(pd.read_csv(ROOT / "C_angle2_forest/outputs/angle2_models.csv"))

## Full retrain (optional, ~10 min)

This overwrites `C_angle2_forest/outputs/` and may leave partial files if interrupted. Use a fresh clone/copy, or restore with `git restore C_angle2_forest/outputs` after a failed run. Order matters: `evaluate_rf.py` creates `metrics.csv`, which `finalize_rf.py` needs.

In [ ]:
run(["code/tune_rf.py"], cwd="C_angle2_forest")
run(["code/evaluate_rf.py"], cwd="C_angle2_forest")
run(["code/evaluate_conditional.py"], cwd="C_angle2_forest")
run(["code/finalize_rf.py"], cwd="C_angle2_forest")

## Merge four models (needs B results)

Uses the included `B_angle2_linear/results/` and the C outputs.

In [ ]:
run(["code/merge_angle2.py"], cwd="C_angle2_forest")

## Optional diagnostic steps not in the main chain

`selection_check.py` audits leave-one-fold selection stability; `figures.py` regenerates the six figures. Both use the outputs created above and are optional.


In [ ]:
run(["code/selection_check.py"], cwd="C_angle2_forest")

In [ ]:
run(["code/figures.py"], cwd="C_angle2_forest")